# Chạy lô ĐA1 trên Colab L4: embed + index + thử truy xuất

Đầu vào lấy từ Drive `MyDrive/bv-rag/processed/{VER}/` (đẩy lên từ WSL bằng `scripts/sync.sh push processed v1`).
Đầu ra ghi về Drive `vectors/{VER}/{structure,fixed}/` (`dense.npy`, `sparse.jsonl`, `chunk_ids.json`, `meta.json`).
Notebook chỉ điều phối; logic nằm trong `src/`.

1. **Runtime → Change runtime type → GPU L4 → Save → Connect.** Cell 1 dừng nếu GPU không phải L4.
2. Sửa `VER`, `BRANCH` ở cell 2 nếu cần (code chưa merge thì đặt `BRANCH` = tên nhánh).
3. **Runtime → Run all.** Cell cuối hỏi xác nhận rồi tắt runtime.
4. Trước khi commit notebook: Edit → Clear all outputs.

Chỉ tài liệu công khai và dữ liệu giả được đưa lên Drive/Colab.

## 1. Kiểm tra GPU (bắt buộc L4)

In [ ]:
import subprocess
GPU = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).decode().strip()
print(GPU)
assert "L4" in GPU, f"GPU là {GPU}, không phải L4: đổi runtime trước mọi lần chạy chính thức (decision-log 2026-10-25)"

## 2. Tham số, mount Drive

In [ ]:
import os
VER = "v1"                   # phiên bản dữ liệu (processed/vN, vectors/vN)
BRANCH = "main"              # nhánh code
CHUNKING = ["structure", "fixed"]

from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/bv-rag"
os.environ["DRIVE_DIR"], os.environ["QDRANT_URL"] = DRIVE_DIR, ":memory:"
!ls -la {DRIVE_DIR}/processed/{VER}

## 3. Clone repo, cài thư viện
Torch bản CUDA có sẵn trên Colab. Nếu pip báo cần khởi động lại: Runtime → Restart session, rồi chạy tiếp từ cell 2.

In [ ]:
!rm -rf /content/bv-rag && git clone -q -b {BRANCH} https://github.com/daisyngocha/bv-rag.git /content/bv-rag
%cd /content/bv-rag
!git log --oneline -1
!pip -q install -r requirements/base.txt
import sys; sys.path.insert(0, "/content/bv-rag")

## 4. Embed bằng bge-m3
Ghi thẳng vào Drive. `s_per_chunk` trên L4 nên quanh 0,026 (smoke v4).

In [ ]:
from src.retrieve.encoder import M3Encoder
from src.ingest.indexer import CHUNK_FILES, embed_corpus

enc = M3Encoder()
for ch in CHUNKING:
    meta = embed_corpus(f"{DRIVE_DIR}/processed/{VER}/{CHUNK_FILES[ch]}", f"{DRIVE_DIR}/vectors/{VER}/{ch}", enc)
    print(f"{ch}: {meta['n']} chunk, {meta['encode_s']} s ({meta['s_per_chunk']} s/chunk), "
          f"GPU {meta['gpu']}, commit {meta['repo_commit']}")

## 5. Nạp Qdrant (:memory:) và thử truy xuất
Kiểm tra nhanh bằng mắt, chưa phải đánh giá. So top-5 giữa chunk cấu trúc và cố định, dense và hybrid.

In [ ]:
from src.ingest.indexer import build_index, read_jsonl
from src.retrieve.hybrid import Retriever, collection_name, get_client

client = get_client()
rets = {}
for ch in CHUNKING:
    chunks = read_jsonl(f"{DRIVE_DIR}/processed/{VER}/{CHUNK_FILES[ch]}")
    col = collection_name(VER, ch)
    print(col, build_index(client, col, chunks, f"{DRIVE_DIR}/vectors/{VER}/{ch}"), "điểm")
    rets[ch] = Retriever(client, col, enc, top_k=5)

QUESTIONS = [
    "Thời gian chờ đối với bệnh đặc biệt là bao lâu?",
    "Bảo Việt An Gia nhận người được bảo hiểm từ bao nhiêu tuổi?",
    "Quyền lợi thai sản được chi trả những gì?",
    "Hồ sơ yêu cầu bồi thường gồm những giấy tờ nào?",
    "Trẻ em tham gia một mình thì đồng chi trả thế nào?",
    "Chi phí điều trị nội trú tại ICU được chi trả bao nhiêu?",
]
for q in QUESTIONS:
    print(f"\n## {q}")
    for ch, r in rets.items():
        for mode in ("dense", "hybrid"):
            hits = r.search(q, mode=mode)
            print(f"  {ch:9s} {mode:6s} " + ", ".join(f"{h['chunk_id']}({h['score']:.3f})" for h in hits))

## 6. Xem nội dung 1 chunk

In [ ]:
CID = "BVP_QT_C1_M1_K1"
for ch in CHUNKING:
    for c in read_jsonl(f"{DRIVE_DIR}/processed/{VER}/{CHUNK_FILES[ch]}"):
        if c["chunk_id"] == CID:
            print(c["text"])

## 7. Tắt runtime
Runtime vẫn tính unit khi không chạy gì. Vector đã nằm trên Drive.

In [ ]:
from google.colab import runtime
if input("Tắt runtime? (y/n) ").strip().lower() == "y":
    runtime.unassign()